In [1]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.optimizers import Adam
import keras
from keras.models import Sequential, Model
from keras.layers import *
from keras.utils import Sequence
from keras.layers import Conv2D, MaxPooling2D
from qkeras import *

from keras.utils import Sequence
from keras.callbacks import CSVLogger
from keras.callbacks import EarlyStopping

import os
import random
from datetime import datetime
import time

import matplotlib.pyplot as plt

pi = 3.14159265359
maxval=1e9
minval=1e-9

2026-07-16 19:13:01.384511: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
from DG.OptimizedDataGenerator_v2p5 import OptimizedDataGenerator
from loss import custom_loss
from SoftQuantizeLayer import SoftQuantizeLayer
from AnnealingScheduler import AnnealingScheduler
# from models.models import CreateModel # Conv2D model

In [3]:
def create_tfrecords(NOISE_MU=0.0, NOISE_SIGMA=0.0):

    dataset_base_dir = "/uscms/home/jennetd/nobackup/smart-pixels/noise-paper/"
    tfrecords_base_dir = "/uscms/home/jennetd/nobackup/smart-pixels/tfrecords/"

    dataset_dir_test   = os.path.join(dataset_base_dir, "dataset_2s_16x16_50x12P5_centeredIncidence_parquets/", 'test_contained/')
    tfrecords_dir_test = os.path.join(tfrecords_base_dir, "TFR_test",'2s_16x16_'+str(int(NOISE_SIGMA))+'eNoise_test')

    batch_size = 5000
    test_batch_size = 5000
    test_file_size = len(os.listdir(dataset_dir_test))

    start_time = time.time()
    test_generator = OptimizedDataGenerator(
        dataset_base_dir = dataset_dir_test,
        file_type = "parquet",
        data_format = "3D",
        batch_size = test_batch_size,
        # optimize_batch_size = True,
        file_count = test_file_size,
        to_standardize= False,
        select_contained = True,
        noise = [NOISE_MU,NOISE_SIGMA], #[mean, sigma]
        min_threshold = None,
        max_threshold = None,
        labels_list = ['x-midplane','y-midplane','cotAlpha','cotBeta'],
        input_shape = (2,16,16), # (20,13,21),
        transpose = (0,2,3,1),
        shuffle = False, 
        files_from_end=True,
        tfrecords_dir = tfrecords_dir_test,
        use_time_stamps = [0,19],
        max_workers = 2
    )

In [4]:
create_tfrecords(0.0, 0.0)

Processing Files...: 100%|██████████| 100/100 [00:32<00:00,  3.10it/s]


Directory /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_0eNoise_test does not exist and cannot be removed.


Saving batches as TFRecords: 100%|██████████| 278/278 [01:23<00:00,  3.33it/s]


Metadata saved successfully ast /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_0eNoise_test/metadata.json
Loading metadata from /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_0eNoise_test/metadata.json


In [5]:
create_tfrecords(0.0, 40.0)

Processing Files...: 100%|██████████| 100/100 [00:42<00:00,  2.36it/s]


Directory /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_40eNoise_test does not exist and cannot be removed.


Saving batches as TFRecords: 100%|██████████| 278/278 [01:21<00:00,  3.41it/s]


Metadata saved successfully ast /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_40eNoise_test/metadata.json
Loading metadata from /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_40eNoise_test/metadata.json


In [6]:
create_tfrecords(0.0, 80.0)

Processing Files...: 100%|██████████| 100/100 [00:46<00:00,  2.17it/s]


Directory /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_80eNoise_test does not exist and cannot be removed.


Saving batches as TFRecords: 100%|██████████| 278/278 [01:29<00:00,  3.09it/s]


Metadata saved successfully ast /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_80eNoise_test/metadata.json
Loading metadata from /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_80eNoise_test/metadata.json


In [7]:
create_tfrecords(0.0, 120.0)

Processing Files...: 100%|██████████| 100/100 [00:42<00:00,  2.35it/s]


Directory /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_120eNoise_test does not exist and cannot be removed.


Saving batches as TFRecords: 100%|██████████| 278/278 [01:27<00:00,  3.18it/s]


Metadata saved successfully ast /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_120eNoise_test/metadata.json
Loading metadata from /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_120eNoise_test/metadata.json


In [8]:
create_tfrecords(0.0, 240.0)

Processing Files...: 100%|██████████| 100/100 [00:48<00:00,  2.08it/s]


Directory /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_240eNoise_test does not exist and cannot be removed.


Saving batches as TFRecords: 100%|██████████| 278/278 [01:18<00:00,  3.54it/s]


Metadata saved successfully ast /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_240eNoise_test/metadata.json
Loading metadata from /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_240eNoise_test/metadata.json


In [4]:
def create_tfrecords_slim(NOISE_MU=0.0, NOISE_SIGMA=0.0):

    dataset_base_dir = "/uscms/home/jennetd/nobackup/smart-pixels/noise-paper/"
    tfrecords_base_dir = "/uscms/home/jennetd/nobackup/smart-pixels/tfrecords/"

    dataset_dir_test   = os.path.join(dataset_base_dir, "dataset_2s_16x16_50x12P5_centeredIncidence_parquets", 'test_contained/')

    tfrecords_dir_test   = os.path.join(tfrecords_base_dir, "TFR_test",'2s_16x16_'+str(int(NOISE_SIGMA))+'eN_raw_slim')

    batch_size = 5000
    test_batch_size = 5000
    test_file_size = len(os.listdir(dataset_dir_test))

    start_time = time.time()
    test_generator = OptimizedDataGenerator(
        dataset_base_dir = dataset_dir_test,
        file_type = "parquet",
        data_format = "3D",
        batch_size = test_batch_size,
        # optimize_batch_size = True,
        file_count = test_file_size,
        to_standardize= False,
        select_contained = True,
        noise = [NOISE_MU,NOISE_SIGMA], #[mean, sigma]
        min_threshold = None,
        max_threshold = None,
        labels_list = ['x-midplane','y-midplane','cotBeta'],
        input_shape = (2,16,16), # (20,13,21),
        transpose = (0,2,3,1),
        shuffle = False, 
        files_from_end=True,
        tfrecords_dir = tfrecords_dir_test,
        use_time_stamps = [0,19],
        max_workers = 2
    )

In [5]:
create_tfrecords_slim(0.0, 0.0)

Processing Files...: 100%|██████████| 100/100 [01:33<00:00,  1.07it/s]


Directory /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_0eN_raw_slim is removed...


Saving batches as TFRecords: 100%|██████████| 278/278 [01:50<00:00,  2.51it/s]


Metadata saved successfully ast /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_0eN_raw_slim/metadata.json
Loading metadata from /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_0eN_raw_slim/metadata.json


In [6]:
create_tfrecords_slim(0.0, 40.0)

Processing Files...: 100%|██████████| 100/100 [00:55<00:00,  1.81it/s]


Directory /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_40eN_raw_slim is removed...


Saving batches as TFRecords: 100%|██████████| 278/278 [01:40<00:00,  2.77it/s]


Metadata saved successfully ast /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_40eN_raw_slim/metadata.json
Loading metadata from /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_40eN_raw_slim/metadata.json


In [7]:
create_tfrecords_slim(0.0, 80.0)

Processing Files...: 100%|██████████| 100/100 [00:53<00:00,  1.88it/s]


Directory /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_80eN_raw_slim is removed...


Saving batches as TFRecords: 100%|██████████| 278/278 [01:27<00:00,  3.17it/s]


Metadata saved successfully ast /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_80eN_raw_slim/metadata.json
Loading metadata from /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_80eN_raw_slim/metadata.json


In [8]:
create_tfrecords_slim(0.0, 120.0)

Processing Files...: 100%|██████████| 100/100 [00:50<00:00,  1.96it/s]


Directory /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_120eN_raw_slim is removed...


Saving batches as TFRecords: 100%|██████████| 278/278 [01:14<00:00,  3.75it/s]


Metadata saved successfully ast /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_120eN_raw_slim/metadata.json
Loading metadata from /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_120eN_raw_slim/metadata.json


In [9]:
create_tfrecords_slim(0.0, 240.0)

Processing Files...: 100%|██████████| 100/100 [00:51<00:00,  1.93it/s]


Directory /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_240eN_raw_slim is removed...


Saving batches as TFRecords: 100%|██████████| 278/278 [01:15<00:00,  3.68it/s]


Metadata saved successfully ast /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_240eN_raw_slim/metadata.json
Loading metadata from /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_test/2s_16x16_240eN_raw_slim/metadata.json
